# Estimación de la Demanda de Consumo en EE.UU. (1987T4 – 2019T4 / 2024T2)

El objetivo del estudio es estimar una función de consumo trimestral para Estados Unidos, utilizando datos de la base FRED (Federal Reserve Bank of St. Louis).  
Se plantea un modelo dinámico con variables macroeconómicas clave y dummies para capturar los efectos del Covid-19 (2020T1 – 2021T2).

La especificación general es:

Δ ln(C_t) = α + β1 ln(Y_t-1) – β2 ln(C_t-1) + β3 ln(W_t) + β4 ln(AL_t-1) + β5 Δ ln(C_t-1) + … + δ (DUM_Covid) + ε_t

---

## Variables del modelo

- **C_t: Consumo privado real**  
  Medido en miles de millones de USD, ajustado por inflación.  
  Fuente: Personal Consumption Expenditures (PCE), FRED.  
  Se utiliza la variación logarítmica Δ ln(C_t) para aproximar la tasa de crecimiento.

- **Y_t-1: Renta disponible real (rezagada un periodo)**  
  Variable fundamental que explica la capacidad de gasto de los hogares.  
  Fuente: Real Disposable Personal Income, FRED.  
  Se introduce en logaritmos y con rezago para captar la relación de causalidad.

- **C_t-1: Consumo del periodo anterior**  
  Captura la inercia y persistencia del consumo en el tiempo.  
  Permite modelar el carácter dinámico del gasto de los hogares.

- **W_t: Riqueza financiera o patrimonial de los hogares**  
  Incluye activos financieros y patrimoniales (acciones, bonos, inmuebles).  
  Refleja el “efecto riqueza” en el consumo.  
  Fuente: Household Net Worth, FRED.

- **AL_t-1: Activos líquidos (rezagados un periodo)**  
  Representados por agregados monetarios como M3.  
  Miden la disponibilidad inmediata de liquidez para financiar consumo.  
  Se corrigen por inflación para obtener valores reales.

- **Δ ln(C_t-1): Cambio en el consumo del trimestre anterior**  
  Introducido como variable dinámica para recoger efectos de aceleración o freno en la evolución del gasto.

- **DUM_Covid: Variables dummy (2020T1 – 2021T2)**  
  Seis dummies trimestrales que toman valor 1 en el trimestre correspondiente y 0 en los demás.  
  Capturan el efecto extraordinario de la pandemia sobre el consumo.

- **ε_t: Término de error**  
  Recoge factores no observados que afectan al consumo.

---

## Concepto general del estudio

El análisis busca comprender los determinantes del consumo privado en EE.UU. desde finales de los 80 hasta la actualidad, incorporando:

- Efecto renta disponible (determinante clave del consumo).  
- Persistencia e inercia del consumo pasado.  
- Efecto riqueza y liquidez como factores financieros que influyen en las decisiones de gasto.  
- Impacto del Covid-19 a través de shocks temporales capturados con dummies.  

Se trata de un modelo de corte macroeconómico y econométrico dinámico, que permitirá estimar elasticidades de consumo y analizar cómo distintos factores explican su evolución en el largo y corto plazo.



## Importación de Librerías


In [94]:
# ---- 0) Imports
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.stats.api as sms
from statsmodels.stats.diagnostic import het_white, acorr_ljungbox
from statsmodels.tsa.stattools import adfuller
import openpyxl

## Tratamiento de los Datos

In [95]:
# Cargar los datos desde el el Excel
file_path = 'data_c_eeuu.xlsx'  # Ajusta la ruta
df = pd.read_excel(file_path) 
df = df.dropna().copy()
df.head()
df.isnull().sum()


t              0
delta_ln_ct    0
y_t^d          0
r_t^cp         0
r_t^lp         0
delta_u_t      0
gap_y^d        0
AL_t-1         0
IA_t-1         0
ln_yt          0
ln_ct-1        0
dtype: int64

In [96]:
df.head()

,t,delta_ln_ct,y_t^d,r_t^cp,r_t^lp,delta_u_t,gap_y^d,AL_t-1,IA_t-1,ln_yt,ln_ct-1
3,1987-10-01,8.676222,6659.742,3.166814,4.687983,-0.166667,112.13600,24.687481,17926176.0,8.803836,8.673405
4,1988-01-01,8.694016,6769.320,2.676322,4.248831,-0.133333,158.50025,24.879714,18336577.0,8.820156,8.676222
5,1988-04-01,8.702350,6846.743,2.946691,4.521236,-0.233333,136.52875,24.807003,18643587.0,8.831528,8.694016
6,1988-07-01,8.711342,6916.391,3.038948,4.561986,0.000000,118.34200,24.753948,19122046.0,8.841649,8.702350
7,1988-10-01,8.724579,6982.304,3.144644,4.479552,-0.133333,103.61450,24.612535,19528780.0,8.851134,8.711342


In [97]:
# ---- 2) Convertir las fechas a formato datetime
df['t'] = pd.to_datetime(df['t'], format='%Y-%m-%d')

# ---- 4) Crear dummies Covid (2020T1 - 2021T2)
# Considera las fechas específicas de los trimestres 2020T1 a 2021T2 para crear las dummies

covid_periods = [
    '2020-01-01', '2020-04-01', '2020-07-01', '2020-10-01',
    '2021-01-01', '2021-04-01', '2021-07-01'
]

df['AL/y'] = df['AL_t-1'] / df['y_t^d']  # Tasa de desempleo
df['IA/y'] = df['IA_t-1'] / df['y_t^d']  # Inversión bruta en bienes de equipo sobre PIB
df['y-c'] = df['ln_yt'] - df['ln_ct-1']  # Ahorro

# Asignar las dummies
for date in covid_periods:
    df[date] = np.where(df['t'] == date, 1, 0)

## Construcción del Modelo

In [98]:
# Regresión propuesta con todas las variables independientes
X = df[['y-c', 'r_t^cp', 'r_t^lp', 'delta_u_t', 'gap_y^d', 'AL/y', 'IA/y'] + covid_periods]  # Variables independientes
y = df['delta_ln_ct']  # Variable dependiente

# Añadir constante para el modelo (intercepto)
X = sm.add_constant(X)

#6) Estimación del modelo (OLS)
model = sm.OLS(y, X, missing='drop')  # Estimación por OLS 
results = model.fit()
print(results.summary())
# Para verlo en Latex 
with open("resultados_regresion.tex", "w") as f:
    f.write(results.summary().as_latex())

                            OLS Regression Results                            
Dep. Variable:            delta_ln_ct   R-squared:                       0.984
Model:                            OLS   Adj. R-squared:                  0.983
Method:                 Least Squares   F-statistic:                     578.3
Date:                Thu, 25 Sep 2025   Prob (F-statistic):          5.06e-109
Time:                        11:42:17   Log-Likelihood:                 279.85
No. Observations:                 144   AIC:                            -529.7
Df Residuals:                     129   BIC:                            -485.1
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          9.0322      0.031    287.952      0.0

In [99]:
# Test de heterocedasticidad (White)
white_test = het_white(results.resid, results.model.exog)
print("White Test for Heteroscedasticity:")
print(white_test)

White Test for Heteroscedasticity:
(51.13947298509392, 0.13326267898506558, 1.370065779755247, 0.1034949393058685)


Como podemos observar, ninguno de los p-valores es menor que el p=0.05. Por tanto no podemos rechazar la hipotesis H0: 'Existencia de homocedasticidad en los residuos'. 

In [100]:
# Test de autocorrelación 
from statsmodels.stats.stattools import durbin_watson
dw = durbin_watson(results.resid)
print("Durbin-Watson:", dw)


Durbin-Watson: 0.6660986489662213


In [101]:
from statsmodels.stats.diagnostic import acorr_breusch_godfrey
bg_test = acorr_breusch_godfrey(results, nlags=4)  
print("Breusch-Godfrey Test:", bg_test)


Breusch-Godfrey Test: (71.34051446403898, 1.1828184018387332e-14, 30.68272587613959, 8.665070095929285e-18)


Mediante ambos métodos encontramos que existe autocorrelación positiva de los errores. Esto no implica que los coeficientes estén sesgados. Sin embargo, sí que puede hacer que los p-valores y los estadísticos t y F no sean los adeacuados. Por tanto, realizaremos el modelo con errores corregidos para podernos adaptar a la situación de autocorrelación

## Modelo con Errores HAC 

In [ ]:
results_hac = results.get_robustcov_results(cov_type='HAC', maxlags=4)
print(results_hac.summary())
jupyter nbconvert --to html prueba1_ej3.ipynb



                            OLS Regression Results                            
Dep. Variable:            delta_ln_ct   R-squared:                       0.984
Model:                            OLS   Adj. R-squared:                  0.983
Method:                 Least Squares   F-statistic:                     420.1
Date:                Thu, 25 Sep 2025   Prob (F-statistic):           1.84e-88
Time:                        11:42:17   Log-Likelihood:                 279.85
No. Observations:                 144   AIC:                            -529.7
Df Residuals:                     129   BIC:                            -485.1
Df Model:                          14                                         
Covariance Type:                  HAC                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          9.0322      0.053    171.201      0.0

/Users/miguelfernandez/Desktop/Programación/Universidad/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 14, but rank is 8
  warnings.warn('covariance of constraints does not have full '


In [ ]:
# Guardamos los resultados con errores HAC en formato LaTeX
with open("resultados_regresion_hac.tex", "w") as f:
    f.write(results_hac.summary().as_latex())



In [107]:
model


## SEE YOU SOON 
